# 3DEP Extras — Terrain Feature Factory

Build reusable terrain features from the USGS 3DEP 10m DEM for any AOI.

In [ ]:
import ee, numpy as np, pandas as pd

ee.Authenticate()
ee.Initialize(project="shrubwise-dc-488219")

THREEDEP_10M = ee.ImageCollection("USGS/3DEP/10m_collection")
AOI_BLISS = ee.Geometry.Rectangle([-120.1018846, 38.99274873, -120.0899834, 39.0020357], geodesic=False)


In [ ]:
def threedep_mosaic() -> ee.Image:
    return THREEDEP_10M.mosaic().select("elevation").toFloat().rename("elevation")

def terrain_feature_cube(*, smooth=False) -> ee.Image:
    dem = threedep_mosaic()
    if smooth:
        dem = dem.focal_mean(radius=1, units="pixels").rename("elevation")

    slope = ee.Terrain.slope(dem).rename("slope_deg")
    aspect = ee.Terrain.aspect(dem).rename("aspect_deg")

    # Northness / eastness are often easier for models than raw circular aspect
    aspect_rad = aspect.multiply(np.pi / 180.0)
    northness = aspect_rad.cos().rename("northness")
    eastness = aspect_rad.sin().rename("eastness")

    # Simple local relief proxy at 3 scales
    relief_3 = dem.focal_max(1, "pixels").subtract(dem.focal_min(1, "pixels")).rename("relief_3px")
    relief_7 = dem.focal_max(3, "pixels").subtract(dem.focal_min(3, "pixels")).rename("relief_7px")
    relief_15 = dem.focal_max(7, "pixels").subtract(dem.focal_min(7, "pixels")).rename("relief_15px")

    return ee.Image.cat([dem, slope, aspect, northness, eastness, relief_3, relief_7, relief_15])

cube = terrain_feature_cube(smooth=False)
print(cube.bandNames().getInfo())


In [ ]:
def summarize_feature_cube(image: ee.Image, aois: dict, *, scale=10):
    reducer = (ee.Reducer.count()
               .combine(ee.Reducer.mean(), sharedInputs=True)
               .combine(ee.Reducer.median(), sharedInputs=True)
               .combine(ee.Reducer.min(), sharedInputs=True)
               .combine(ee.Reducer.max(), sharedInputs=True))
    rows = []
    for name, geom in aois.items():
        stats = image.reduceRegion(reducer, geom, scale, maxPixels=1e9, bestEffort=True).getInfo()
        row = {"aoi": name, **stats}
        rows.append(row)
    return pd.DataFrame(rows).set_index("aoi")

AOIS = {"DL_Bliss": AOI_BLISS}
summarize_feature_cube(cube, AOIS)
